# Audio model tuning and evaluation
Five models: Decision Tree, RBF SVM, custom CNN, Keras Applications ImageNet ResNet50 (warmup + top-stage fine-tuning), Transformer–CNN. Tuning uses validation only. Champion maximizes abnormal recall subject to validation specificity ≥0.5. Test is untouched until selection. No outputs are prefilled.

In [ ]:
from pathlib import Path
import os, subprocess, sys
if not Path("training").exists(): os.chdir(Path.cwd().parent)
# Install the optional deep-learning extra before running all five.
subprocess.run([sys.executable, "-m", "training.benchmark", "--epochs", "20", "--trials", "6", "--activate"], check=True)

In [ ]:
import json, pandas as pd
path = max(Path("vault/runs").glob("*/benchmark.json"), key=lambda p:p.stat().st_mtime)
report = json.loads(path.read_text())
print("Run:",report["run_id"],"Complete:",report["complete_five_model_benchmark"],"Subset:",report["subset"])
table = pd.DataFrame([{ "model": m["name"], **m["test"] } for m in report["models"]]).set_index("model")
display(table)
table[["recall_abnormal", "specificity", "auprc"]].plot.bar(ylim=(0,1), title="Held-out test performance")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
fig, axes = plt.subplots(1,len(report["models"]),figsize=(5*len(report["models"]),4),squeeze=False)
for axis, model in zip(axes[0],report["models"]):
    m=model["test"]
    ConfusionMatrixDisplay([[m["tn"],m["fp"]],[m["fn"],m["tp"]]],display_labels=["normal","abnormal"]).plot(ax=axis,colorbar=False)
    axis.set_title(model["name"])
plt.tight_layout()